# 07 · Shuffle, Wide vs. Narrow, Broadcast Join 

**Teoria**: docs/03-transformacoes-acoes-dag.md, docs/06-persistencia-e-otimizacao.md

**Pré-requisito**: `make up-cluster` ainda em execução.

🎯 **Objetivo**: entender na prática a diferença entre transformações **narrow** (sem shuffle) 
e **wide** (com shuffle), e como o broadcast join elimina o shuffle do lado grande.

Este laboratório coloca um join sem Shuffle (`empresas`, 50 linhas → `broadcast()`) 
lado a lado com um com muito Shuffle (`funcionarios`, milhares de 
linhas → SortMergeJoin), para que você veja a diferença de custo na 
Spark UI, não apenas na teoria.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15002")   # Endpoint gRPC do servidor Spark Connect
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4040")

🖥️  Master UI:    http://localhost:8080
🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082
📊 Spark App UI:  http://localhost:4040


In [4]:
# Lê cada tabela da camada Bronze em formato Parquet (colunar, comprimido)
# Criar Spark Data Frame
sdf_empresas = spark.read.parquet("/data/bronze/empresas")
sdf_funcionarios = spark.read.parquet("/data/bronze/funcionarios")
sdf_vendas = spark.read.parquet("/data/bronze/vendas")

## Broadcast join — sem Shuffle de `vendas`

📌 Procure por `BroadcastHashJoin` no plano abaixo. Depois verifique a aba 
Stages da Spark UI (http://localhost:4040) para este Job: nenhum nó `Exchange` para o 
lado grande.

🧠 **Como funciona**: o Spark copia a tabela `empresas` (50 linhas) para a memória de 
cada executor. O join acontece localmente, sem movimento de dados do lado `vendas`.

In [ ]:
from pyspark.sql.functions import broadcast, sum as spark_sum

#  Broadcast join: dica explícita broadcast() — força BroadcastHashJoin
# empresas é pequena (~50 linhas) e será copiada para cada executor
broadcast_join = sdf_vendas.join(broadcast(sdf_empresas), "id_empresa")

# explain() mostra o plano físico — procure por BroadcastHashJoin
broadcast_join.explain()

# Executa o join com groupBy para materializar o resultado
# Obs.: o count() no final força a execução
sdf = broadcast_join.groupBy("setor").agg(spark_sum("valor"))
print(f"Total: {sdf.count()}")
sdf.show()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id_empresa#3264, id_venda#3262L, id_funcionario#3263L, valor#3265, dia#3266, ano#3267, mes#3268, nome_empresa#3277, setor#3278, regiao#3279]
   +- BroadcastHashJoin [id_empresa#3264], [id_empresa#3276], Inner, BuildRight, false
      :- Filter isnotnull(id_empresa#3264)
      :  +- FileScan parquet [id_venda#3262L,id_funcionario#3263L,id_empresa#3264,valor#3265,dia#3266,ano#3267,mes#3268] Batched: true, DataFilters: [isnotnull(id_empresa#3264)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/data/bronze/vendas], PartitionFilters: [], PushedFilters: [IsNotNull(id_empresa)], ReadSchema: struct<id_venda:bigint,id_funcionario:bigint,id_empresa:int,valor:double,dia:int>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=1254]
         +- Filter isnotnull(id_empresa#3276)
            +- FileScan parquet [id_empresa#3276,nome_empresa#3277,setor#3278,

📌 **Resultado do broadcast join**:

O plano deve mostrar `BroadcastHashJoin` sem `Exchange` no lado `vendas`. 
Isso significa ZERO shuffle de dados — apenas uma cópia pequena de `empresas` 
para cada worker.

💡 **Dica**: na aba **Stages** da Spark UI, compare o número de Stages e Tasks 
deste Job com o próximo (shuffle join). O broadcast join tipicamente tem menos Stages.

## Shuffle join — ambos os lados são redistribuídos

📌 Procure por `SortMergeJoin` e (geralmente) dois nós `Exchange` — um para cada lado do 
join. Na Spark UI, compare a duração deste Stage contra o 
broadcast join acima.

🧠 **Como funciona**: o Spark reparticiona AMBOS os DataFrames por `id_funcionario`. 
Isso exige shuffle dos dois lados — muito mais caro em termos de rede e disco.

In [8]:
# Shuffle join: sem broadcast — Catalyst escolhe SortMergeJoin
# Funcionarios tem milhares de linhas, não cabe no limite de broadcast
shuffle_join = sdf_vendas.join(sdf_funcionarios, "id_funcionario")

# explain() mostra os nós Exchange — um para vendas, outro para funcionarios
shuffle_join.explain()

# Executa com groupBy para materializar
shuffle_join.groupBy("cargo").agg(spark_sum("valor")).count()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id_funcionario#3847L, id_venda#3846L, id_empresa#3848, valor#3849, dia#3850, ano#3851, mes#3852, nome_funcionario#3861, id_empresa#3862, cargo#3863, salario#3864, data_admissao#3865]
   +- BroadcastHashJoin [id_funcionario#3847L], [id_funcionario#3860L], Inner, BuildRight, false
      :- Filter isnotnull(id_funcionario#3847L)
      :  +- FileScan parquet [id_venda#3846L,id_funcionario#3847L,id_empresa#3848,valor#3849,dia#3850,ano#3851,mes#3852] Batched: true, DataFilters: [isnotnull(id_funcionario#3847L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/data/bronze/vendas], PartitionFilters: [], PushedFilters: [IsNotNull(id_funcionario)], ReadSchema: struct<id_venda:bigint,id_funcionario:bigint,id_empresa:int,valor:double,dia:int>
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=1662]
         +- Filter isnotnull(id_funcionario#3860L)
            +- FileS

6

📌 **Resultado do shuffle join**:

O plano mostra `SortMergeJoin` com dois `Exchange` (um para cada lado). 
Cada `Exchange` representa um shuffle — dados sendo reescritos em disco e transferidos 
pela rede entre executores.

⚠️ **Atenção**: o shuffle join pode ser 10×-100× mais lento que o broadcast join 
dependendo do volume de dados. Na Spark UI, veja a diferença na métrica 
**Shuffle Read/Write** (MB transferidos).

## Reparticionando uma vez, reutilizando entre operações

🧠 **Estratégia**: se você sabe que executará várias consultas `groupBy("id_empresa")` 
em sequência, repartitionar por essa chave antecipadamente evita pagar o custo do 
Shuffle mais de uma vez.

📌 O `repartition(8, "id_empresa")` faz um shuffle único e caro, mas os groupBy 
subsequentes aproveitam a partição já alinhada — sem novo shuffle.

In [9]:
# Reparticiona por id_empresa em 8 partições — shuffle único e proposital
# Após isso, todos os dados com o mesmo id_empresa estão na MESMA partição
vendas_by_empresa = sdf_vendas.repartition(8, "id_empresa")
vendas_by_empresa.cache()                         # Cacheia para reuso
vendas_by_empresa.count()                         # Materializa o cache

# Ambos os groupBy reusam o mesmo particionamento — sem shuffle extra
vendas_by_empresa.groupBy("id_empresa").agg(spark_sum("valor")).show()
vendas_by_empresa.groupBy("id_empresa").count().show()

# Libera o cache — importante para não reter memória desnecessária
vendas_by_empresa.unpersist()

+----------+--------------------+
|id_empresa|          sum(valor)|
+----------+--------------------+
|        38| 1.736338339000016E7|
|        18| 1.770262472999985E7|
|        14|1.8518668809999805E7|
|        46|1.7886572500000365E7|
|        13|1.8570815510000203E7|
|        12| 1.740025281000004E7|
|        23|1.7292945999999978E7|
|        39|1.7791071070000492E7|
|        16|1.8270081569999907E7|
|         6|1.7798958670000132E7|
|        17|1.8476142450000077E7|
|         9|1.8715459220000356E7|
|        48| 1.784572496999996E7|
|         5| 1.882784836999967E7|
|        10|1.9226770480000038E7|
|        31| 1.914202866999964E7|
|        49|1.8049584509999808E7|
|         3|1.8705671300000004E7|
|        47|1.8894712500000145E7|
|        26|1.8045838989999972E7|
+----------+--------------------+
only showing top 20 rows

+----------+------+
|id_empresa| count|
+----------+------+
|        38|192585|
|        18|197123|
|        14|204782|
|        46|198447|
|        13|206499

DataFrame[id_venda: bigint, id_funcionario: bigint, id_empresa: int, valor: double, dia: int, ano: int, mes: int]

📌 **Por que isso funciona?**:

Quando você reparticiona por `id_empresa`, o Spark garante que todas as linhas com o 
mesmo `id_empresa` fiquem na mesma partição. Um `groupBy("id_empresa")` subsequente 
consegue agregar localmente, sem precisar de outro shuffle.

💡 **Dica técnica**: isso se chama **partition pruning** combinado com **cache** — uma 
técnica poderosa para pipelines que fazem múltiplas agregações pela mesma chave.

⚠️ **Atenção**: o `repartition()` em si é uma operação **wide** (shuffle). O benefício 
só aparece se você fizer pelo menos 2 operações após ele.

In [ ]:
# Encerra a sessão Spark Connect — libera recursos no cluster
spark.stop()